# Smart Lender EDA

This notebook mirrors the exploration flow from the guide: load the dataset, clean whitespace, audit data quality, visualize correlations, and train a baseline model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('data/loan_approval_dataset.csv')
df.columns = df.columns.str.strip()
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
print('Summary statistics')
display(df.describe())

print('Missing values')
display(df.isnull().sum())

print('Target distribution')
display(df['loan_status'].value_counts(normalize=True).mul(100).round(2))

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).drop(columns=['loan_id'])
corr = numeric_cols.corr()

plt.figure(figsize=(10, 8))
plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(label='Correlation')
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title('Correlation Heatmap of Financial Features')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

feature_columns = [
    'no_of_dependents', 'education', 'self_employed', 'income_annum',
    'loan_amount', 'loan_term', 'cibil_score', 'residential_assets_value',
    'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value'
]

X = df[feature_columns].copy()
X['education'] = X['education'].map({'Graduate': 0, 'Not Graduate': 1})
X['self_employed'] = X['self_employed'].map({'No': 0, 'Yes': 1})
y = df['loan_status'].map({'Approved': 0, 'Rejected': 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(n_estimators=250, random_state=42, class_weight='balanced')),
])
clf.fit(X_train, y_train)
pred = clf.predict(X_test)

print(f'Baseline accuracy: {accuracy_score(y_test, pred) * 100:.2f}%')
print(classification_report(y_test, pred, target_names=['Approved', 'Rejected']))